<div style="display:flex;align-items:center;justify-content:space-between;border-bottom:2px solid #c8962d;padding-bottom:12px;margin-bottom:20px">
  <div><strong>Universidad Externado de Colombia</strong><br>
  <span>Programa de Ciencia de Datos · Machine Learning II</span><br>
  <span>Docente: Wilmer Pineda-Ríos</span></div>
  <img src="../../assets/brand/logo-externado.png" width="190">
</div>

# Semana curricular 3 — Random Forest, importancia y AdaBoost

**Impartida en la semana calendario 4.** Este notebook mantiene una única regla experimental para las dos sesiones.

## 1. Objetivos

- Comparar árbol, Bagging, Random Forest y AdaBoost con los mismos folds.
- Contrastar validación cruzada y OOB.
- Interpretar importancias por impureza y permutación.
- Reservar test hasta finalizar la selección.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    precision_score, recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
pd.set_option("display.max_rows", 30)
pd.set_option("display.precision", 4)

## 2. Caso y auditoría de datos

El objetivo es priorizar clientes para retención. `balanced_accuracy` será principal porque pondera por igual el recall de ambas clases.

In [ ]:
candidates = [
    Path("../../datasets/public/Customer_Churn_Iran.csv"),
    Path("datasets/public/Customer_Churn_Iran.csv"),
    Path("../datasets/public/Customer_Churn_Iran.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("No se encontró Customer_Churn_Iran.csv")

df = pd.read_csv(data_path)
target = "Churn"
X = df.drop(columns=target)
y = df[target].astype(int)

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "recall": "recall",
    "precision": "precision",
}

pd.DataFrame({
    "filas": [len(df)],
    "predictores": [X.shape[1]],
    "churn_n": [int(y.sum())],
    "churn_pct": [100 * y.mean()],
})

In [ ]:
df.isna().sum().to_frame('faltantes').T

In [ ]:
y.value_counts().rename(index={0: 'no churn', 1: 'churn'}).to_frame('n').assign(porcentaje=lambda d: 100*d['n']/len(y))

## 3. Árbol, Bagging y Random Forest

Bagging perturba observaciones. Random Forest también restringe las variables candidatas en cada nodo para reducir correlación.

In [ ]:
tree = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=15, random_state=RANDOM_STATE
)
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=None, min_samples_leaf=5),
    n_estimators=300,
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
forest = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    min_samples_leaf=5,
    bootstrap=True,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
stump = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)
adaboost = AdaBoostClassifier(
    estimator=stump,
    n_estimators=200,
    learning_rate=0.5,
    random_state=RANDOM_STATE,
)
models = {
    "Árbol": tree,
    "Bagging": bagging,
    "Random Forest": forest,
    "AdaBoost": adaboost,
}

In [ ]:
rows = []
for name, model in models.items():
    scores = cross_validate(model, X_dev, y_dev, cv=cv, scoring=scoring, n_jobs=-1)
    row = {"modelo": name}
    for metric in scoring:
        values = scores[f"test_{metric}"]
        row[f"{metric}_media"] = values.mean()
        row[f"{metric}_sd"] = values.std(ddof=1)
    rows.append(row)

cv_results = pd.DataFrame(rows).set_index("modelo")
cv_results.sort_values("balanced_accuracy_media", ascending=False)

**Lectura.** La media resume desempeño; la desviación estándar muestra cuánto cambia la conclusión entre particiones.

## 4. Evaluación out-of-bag

`oob_score_` reporta accuracy por defecto. Para que la comparación responda al problema, recalculamos `balanced_accuracy` desde las probabilidades OOB.

In [ ]:
forest.fit(X_dev, y_dev)
oob_probability = forest.oob_decision_function_[:, 1]
valid_oob = ~np.isnan(oob_probability)
oob_prediction = (oob_probability[valid_oob] >= 0.50).astype(int)

pd.Series({
    "oob_accuracy_sklearn": forest.oob_score_,
    "oob_balanced_accuracy": balanced_accuracy_score(y_dev.iloc[valid_oob], oob_prediction),
    "observaciones_con_prediccion": int(valid_oob.sum()),
})

In [ ]:
rf_cv = cv_results.loc["Random Forest", "balanced_accuracy_media"]
pd.Series({
    "RF_balanced_accuracy_CV": rf_cv,
    "RF_balanced_accuracy_OOB": balanced_accuracy_score(y_dev.iloc[valid_oob], oob_prediction),
    "diferencia_absoluta": abs(rf_cv - balanced_accuracy_score(y_dev.iloc[valid_oob], oob_prediction)),
})

## 5. Convergencia del bosque

Aumentar árboles estabiliza la agregación, pero no corrige un protocolo defectuoso.

In [ ]:
convergence = []
for n_trees in [30, 100, 300, 600]:
    model = RandomForestClassifier(
        n_estimators=n_trees, max_features="sqrt", min_samples_leaf=5,
        bootstrap=True, oob_score=True, random_state=RANDOM_STATE, n_jobs=-1,
    ).fit(X_dev, y_dev)
    p = model.oob_decision_function_[:, 1]
    mask = ~np.isnan(p)
    convergence.append({
        "n_estimators": n_trees,
        "oob_balanced_accuracy": balanced_accuracy_score(y_dev.iloc[mask], p[mask] >= 0.5),
        "oob_accuracy": model.oob_score_,
    })
pd.DataFrame(convergence)

## 6. Importancia por impureza

MDI resume cuánto redujo cada variable el criterio de los árboles. Favorece variables con más oportunidades de corte y no demuestra causalidad.

In [ ]:
mdi = pd.Series(forest.feature_importances_, index=X.columns, name="MDI")
mdi.sort_values(ascending=False).head(10).to_frame()

## 7. Importancia por permutación en validación

Usamos el primer fold como demostración: el modelo se ajusta en su train y la permutación ocurre exclusivamente en su validación.

In [ ]:
train_idx, valid_idx = next(cv.split(X_dev, y_dev))
rf_for_importance = RandomForestClassifier(
    n_estimators=300, max_features="sqrt", min_samples_leaf=5,
    random_state=RANDOM_STATE, n_jobs=-1,
).fit(X_dev.iloc[train_idx], y_dev.iloc[train_idx])

perm = permutation_importance(
    rf_for_importance,
    X_dev.iloc[valid_idx], y_dev.iloc[valid_idx],
    scoring="balanced_accuracy", n_repeats=20,
    random_state=RANDOM_STATE, n_jobs=-1,
)
permutation = pd.DataFrame({
    "variable": X.columns,
    "permutation_media": perm.importances_mean,
    "permutation_sd": perm.importances_std,
}).sort_values("permutation_media", ascending=False)
permutation.head(10)

In [ ]:
importance_comparison = (
    mdi.rename_axis("variable").reset_index()
    .merge(permutation, on="variable")
    .sort_values("permutation_media", ascending=False)
)
importance_comparison.head(10)

## 8. Una ronda de AdaBoost a mano

Cinco observaciones parten con peso 0,2. Si el stump falla solo una, $\epsilon=0.2$ y $\alpha=\frac12\log(4)$.

In [ ]:
epsilon = 0.20
alpha = 0.5 * np.log((1-epsilon)/epsilon)
unnormalized = np.array([0.2*np.exp(-alpha)]*4 + [0.2*np.exp(alpha)])
pd.DataFrame({
    "observacion": range(1, 6),
    "acierto": [True, True, True, True, False],
    "peso_sin_normalizar": unnormalized,
    "peso_normalizado": unnormalized/unnormalized.sum(),
})

## 9. Evaluación final

Solo después de escoger por CV se ajusta con todo desarrollo y se consulta test una vez.

In [ ]:
selected_name = cv_results["balanced_accuracy_media"].idxmax()
selected_model = models[selected_name]
selected_model.fit(X_dev, y_dev)
test_pred = selected_model.predict(X_test)

final_metrics = pd.Series({
    "modelo": selected_name,
    "balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
    "accuracy": accuracy_score(y_test, test_pred),
    "precision_churn": precision_score(y_test, test_pred, zero_division=0),
    "recall_churn": recall_score(y_test, test_pred, zero_division=0),
})
final_metrics

In [ ]:
pd.DataFrame(confusion_matrix(y_test, test_pred), index=['real_0','real_1'], columns=['pred_0','pred_1'])

## 10. De lo técnico a la decisión

1. ¿La mejora frente al árbol supera la variabilidad entre folds?
2. ¿Qué error queda concentrado en churn?
3. ¿Coinciden MDI y permutación?
4. ¿Qué riesgo metodológico debe comunicarse?

**Regla de cierre:** importancia predictiva no equivale a efecto causal.